# 08 — Gold-Schicht und Datenqualität

## Zweck
Aus den Silver-Daten **analysebereite Gold-Tabellen** bauen und einen kurzen Qualitätsbericht
erstellen. Notebook `09` liest danach ausschließlich diese Gold-Tabellen — keine Transformationslogik mehr.

## Methodische Trennung
Historische **EEA-Messdaten** und der **Open-Meteo-Live-Snapshot** bleiben getrennt. Jede Tabelle trägt
`dataset_context` (`eea_historical` oder `open_meteo_live`). Beide dürfen nicht direkt verglichen werden
(Messstation vs. Modellwert).

## Ausgabe (5 Parquet-Dateien in `data/gold/`)
`city_air_quality_daily_summary`, `pollutant_ranking_by_city`, `city_context_air_quality`,
`live_air_quality_latest`, `data_quality_summary`.

## Konfiguration und Silver-Daten laden

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)

city_reference = pd.read_parquet(SILVER_DIR / "city_reference.parquet")
city_metadata = pd.read_parquet(SILVER_DIR / "city_metadata.parquet")
eea_daily = pd.read_parquet(SILVER_DIR / "eea_city_daily.parquet")
live_silver = pd.read_parquet(SILVER_DIR / "open_meteo_city_hourly")
print({"eea_daily": len(eea_daily), "live_silver": len(live_silver)})

{'eea_daily': 7938, 'live_silver': 192}


## Gold 1: Tageswerte je Stadt (eea_historical)
Die Silver-Tageswerte werden um den Stadtnamen ergänzt und einheitlich benannt.

In [2]:
daily_summary = (
    eea_daily.merge(city_reference[["city_id", "city_name"]], on="city_id", how="left")
    .rename(columns={"mean_value": "avg_value", "observation_count": "measurement_count"})
    .assign(dataset_context="eea_historical")
    [["city_id", "city_name", "date", "pollutant", "avg_value", "min_value", "max_value",
      "measurement_count", "source", "data_status", "dataset_context"]]
)
daily_summary.to_parquet(GOLD_DIR / "city_air_quality_daily_summary.parquet", index=False)
daily_summary.head()

,city_id,city_name,date,pollutant,avg_value,min_value,max_value,measurement_count,source,data_status,dataset_context
0,amsterdam_nl,Amsterdam,2025-01-01,no2,9.484470,2.1,41.3,264,eea_downloads_api,real_eea_api_postgres,eea_historical
1,amsterdam_nl,Amsterdam,2025-01-01,pm10,16.122876,0.8,227.3,153,eea_downloads_api,real_eea_api_postgres,eea_historical
2,amsterdam_nl,Amsterdam,2025-01-01,pm2_5,13.634167,0.7,205.5,120,eea_downloads_api,real_eea_api_postgres,eea_historical
3,amsterdam_nl,Amsterdam,2025-01-02,no2,27.071212,2.8,84.0,264,eea_downloads_api,real_eea_api_postgres,eea_historical
4,amsterdam_nl,Amsterdam,2025-01-02,pm10,10.191195,2.4,18.5,159,eea_downloads_api,real_eea_api_postgres,eea_historical


## Gold 2: Schadstoff-Rangfolge je Stadt (eea_historical)
Pro Schadstoff der Stadtmittelwert über den Zeitraum, absteigend gereiht.

In [3]:
ranking = (
    daily_summary.groupby(["pollutant", "city_id", "city_name"], as_index=False)
    .agg(avg_value=("avg_value", "mean"),
         days=("date", "nunique"),
         measurement_count=("measurement_count", "sum"))
)
ranking["rank"] = ranking.groupby("pollutant")["avg_value"].rank(ascending=False, method="min").astype(int)
ranking["dataset_context"] = "eea_historical"
ranking = ranking.sort_values(["pollutant", "rank"])
ranking.to_parquet(GOLD_DIR / "pollutant_ranking_by_city.parquet", index=False)
ranking.head(10)

,pollutant,city_id,city_name,avg_value,days,measurement_count,rank,dataset_context
5,no2,rome_it,Rome,25.009749,364,106642,1,eea_historical
3,no2,paris_fr,Paris,22.968982,365,266390,2,eea_historical
4,no2,prague_cz,Prague,22.663012,365,74462,3,eea_historical
7,no2,warsaw_pl,Warsaw,22.553729,365,33928,4,eea_historical
2,no2,madrid_es,Madrid,22.481703,365,281374,5,eea_historical
0,no2,amsterdam_nl,Amsterdam,18.100652,353,85955,6,eea_historical
6,no2,vienna_at,Vienna,16.030066,348,125040,7,eea_historical
1,no2,berlin_de,Berlin,15.331012,365,122310,8,eea_historical
14,pm10,warsaw_pl,Warsaw,21.041079,365,51449,1,eea_historical
12,pm10,prague_cz,Prague,20.736136,365,104348,2,eea_historical


## Gold 3: Rangfolge mit Stadtkontext (eea_historical)
Die Rangfolge wird mit den Wikipedia-Metadaten (Bevölkerungsdichte) verknüpft — als **explorativer**
Kontext, nicht als Erklärung.

In [4]:
context = ranking.merge(
    city_metadata[["city_id", "population", "area_km2", "population_density",
                   "density_comparable", "area_basis_note"]],
    on="city_id", how="left",
)
context.to_parquet(GOLD_DIR / "city_context_air_quality.parquet", index=False)
context.head(10)

,pollutant,city_id,city_name,avg_value,days,measurement_count,rank,dataset_context,population,area_km2,population_density,density_comparable,area_basis_note
0,no2,rome_it,Rome,25.009749,364,106642,1,eea_historical,2746984,1287.36,2133.81,True,
1,no2,paris_fr,Paris,22.968982,365,266390,2,eea_historical,2047602,105.40,19430.00,False,"Kernkommune (20 Arrondissements, ~105 km²) — n..."
2,no2,prague_cz,Prague,22.663012,365,74462,3,eea_historical,1407084,496.21,2835.70,True,
3,no2,warsaw_pl,Warsaw,22.553729,365,33928,4,eea_historical,1862402,517.24,3500.00,True,
4,no2,madrid_es,Madrid,22.481703,365,281374,5,eea_historical,3477497,605.77,5740.60,True,
5,no2,amsterdam_nl,Amsterdam,18.100652,353,85955,6,eea_historical,933680,219.32,5277.00,True,
6,no2,vienna_at,Vienna,16.030066,348,125040,7,eea_historical,2028499,414.78,4890.50,True,
7,no2,berlin_de,Berlin,15.331012,365,122310,8,eea_historical,3596999,891.30,4109.00,True,
8,pm10,warsaw_pl,Warsaw,21.041079,365,51449,1,eea_historical,1862402,517.24,3500.00,True,
9,pm10,prague_cz,Prague,20.736136,365,104348,2,eea_historical,1407084,496.21,2835.70,True,


## Gold 4: Live-Snapshot je Stadt (open_meteo_live)
Aus dem Spark-Streaming-Output je Stadt der **neueste** Messzeitpunkt.

In [5]:
live_silver["event_time_ts"] = pd.to_datetime(live_silver["event_time_ts"], utc=True)
latest_idx = live_silver.groupby("city_id")["event_time_ts"].idxmax()
live_latest = (
    live_silver.loc[latest_idx]
    .assign(dataset_context="open_meteo_live")
    [["city_id", "city_name", "event_time_ts", "pm2_5", "pm10", "no2", "dataset_context"]]
    .reset_index(drop=True)
)
live_latest.to_parquet(GOLD_DIR / "live_air_quality_latest.parquet", index=False)
live_latest

,city_id,city_name,event_time_ts,pm2_5,pm10,no2,dataset_context
0,amsterdam_nl,Amsterdam,2026-06-18 23:00:00+00:00,7.6,10.5,34.6,open_meteo_live
1,berlin_de,Berlin,2026-06-18 23:00:00+00:00,5.8,8.2,12.2,open_meteo_live
2,madrid_es,Madrid,2026-06-18 23:00:00+00:00,13.5,27.1,14.5,open_meteo_live
3,paris_fr,Paris,2026-06-18 23:00:00+00:00,10.7,15.6,29.7,open_meteo_live
4,prague_cz,Prague,2026-06-18 23:00:00+00:00,10.8,14.1,13.1,open_meteo_live
5,rome_it,Rome,2026-06-18 23:00:00+00:00,11.6,18.1,18.2,open_meteo_live
6,vienna_at,Vienna,2026-06-18 23:00:00+00:00,8.2,11.9,12.7,open_meteo_live
7,warsaw_pl,Warsaw,2026-06-18 23:00:00+00:00,12.0,17.0,18.2,open_meteo_live


## Gold 5: Qualitätsbericht
Zeilenzahlen, fehlende Werte und der je Tabelle abgedeckte Zeitraum (`coverage_days`) — die Grundlage,
auf der Notebook `09` seine Aussagen einordnet. Für die EEA-historischen Tabellen ist das die Zahl der
abgedeckten Tage; für den Open-Meteo-Live-Snapshot ist sie **nicht zutreffend** (`NA`), da er eine
Momentaufnahme eines Zeitpunkts ist und keinen historischen Zeitraum abdeckt.

In [6]:
gold_tables = {
    "city_air_quality_daily_summary": daily_summary,
    "pollutant_ranking_by_city": ranking,
    "city_context_air_quality": context,
    "live_air_quality_latest": live_latest,
}
HISTORICAL_DAYS = int(daily_summary["date"].nunique())


def coverage_days(df: pd.DataFrame):
    # Der Open-Meteo-Live-Snapshot ist eine Momentaufnahme, kein historischer
    # Zeitraum -> der abgedeckte Zeitraum ist hier nicht zutreffend (NA).
    if set(df["dataset_context"].unique()) == {"open_meteo_live"}:
        return pd.NA
    return HISTORICAL_DAYS


quality_summary = pd.DataFrame([
    {"table": name, "rows": len(df), "columns": len(df.columns),
     "missing_values": int(df.isna().sum().sum()),
     "coverage_days": coverage_days(df)}
    for name, df in gold_tables.items()
])
quality_summary["coverage_days"] = quality_summary["coverage_days"].astype("Int64")
quality_summary.to_parquet(GOLD_DIR / "data_quality_summary.parquet", index=False)

print({"historische_tage": HISTORICAL_DAYS})
quality_summary

{'historische_tage': 365}


,table,rows,columns,missing_values,coverage_days
0,city_air_quality_daily_summary,7938,11,0,365
1,pollutant_ranking_by_city,22,8,0,365
2,city_context_air_quality,22,13,0,365
3,live_air_quality_latest,8,7,0,<NA>


## Nächster Schritt
Notebook `09` ausführen — Analyse, Visualisierung und Ergebnisgeschichte.